# ⚙️ Notebook 04 — Co-Optimization Engine
**Stock Projection → Gap Detection → PO Recommendation**

> เป้าหมาย: ใช้ M1 + M2 สร้าง actionable PO recommendation table สำหรับ SME

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load models
with open("models/model_m1_demand.pkl","rb") as f: model_m1 = pickle.load(f)
with open("models/model_m2_lead_time.pkl","rb") as f: model_m2 = pickle.load(f)

# Load data
DATA_PATH = "data/raw/"
feature_df = pd.read_csv("data/processed/feature_store.csv", parse_dates=['week'])
po         = pd.read_csv(DATA_PATH + "purchasing_order.csv", parse_dates=["po_date","arrival_date","expire_date"])
stock_mv   = pd.read_csv(DATA_PATH + "stock_movement.csv",   parse_dates=["receive_date","transfer_date"])
products   = pd.read_csv(DATA_PATH + "product_master.csv")
stores     = pd.read_csv(DATA_PATH + "store_master.csv")

TODAY      = pd.Timestamp('2024-12-23')  # สมมติ "วันนี้"
HORIZON    = 4  # forecast 4 สัปดาห์ล่วงหน้า
SERVICE_Z  = 1.65  # 95% service level

print(f"✅ Setup complete | Today: {TODAY.date()} | Horizon: {HORIZON} weeks")

## 1. Current Stock Estimation

In [ ]:
# คำนวณ stock ปัจจุบันต่อ (store, product)
# stock = total received - total sold (approximation)
received = (
    stock_mv[stock_mv['receive_date'] <= TODAY]
    .groupby('store_id')['qty'].sum()
    .reset_index().rename(columns={'qty':'total_received'})
)

sold = (
    pd.read_csv(DATA_PATH + "sales_transaction.csv", parse_dates=["datetime"])
    .query("datetime <= @TODAY")
    .groupby('store_id')['qty'].sum()
    .reset_index().rename(columns={'qty':'total_sold'})
)

current_stock = (
    received.merge(sold, on='store_id', how='left')
)
current_stock['total_sold']     = current_stock['total_sold'].fillna(0)
current_stock['current_stock']  = np.maximum(0, current_stock['total_received'] - current_stock['total_sold'])

print("Current stock per store:")
print(current_stock.to_string(index=False))

## 2. Demand Forecast — Next 4 Weeks

In [ ]:
FEATURE_COLS = [
    'lag_1w','lag_2w','lag_4w','lag_8w',
    'rolling_mean_4w','rolling_std_4w','rolling_mean_8w','rolling_std_8w',
    'has_promo','discount','has_promo_next_week',
    'week_of_year','month','quarter','is_month_end',
    'product_cat_enc','store_type_enc','price_vs_category',
]

# ใช้ข้อมูล 8 สัปดาห์ล่าสุดต่อ product-store เป็น seed
latest = (
    feature_df[feature_df['week'] <= TODAY]
    .sort_values('week')
    .groupby(['store_id','product_id'])
    .tail(1)
    .copy()
)

future_rows = []
for w in range(1, HORIZON+1):
    forecast_week = TODAY + pd.Timedelta(weeks=w)
    rows = latest.copy()
    rows['week']         = forecast_week
    rows['week_of_year'] = forecast_week.isocalendar()[1]
    rows['month']        = forecast_week.month
    rows['quarter']      = (forecast_week.month - 1) // 3 + 1
    rows['is_month_end'] = int(forecast_week.day >= 24)

    valid = rows.dropna(subset=FEATURE_COLS)
    preds = np.maximum(0, model_m1.predict(valid[FEATURE_COLS]))
    valid = valid.copy()
    valid['forecasted_qty'] = preds
    valid['forecast_week']  = forecast_week
    future_rows.append(valid[['store_id','product_id','forecast_week','forecasted_qty']])

forecast_df = pd.concat(future_rows, ignore_index=True)
total_forecast = forecast_df.groupby(['store_id','product_id'])['forecasted_qty'].sum().reset_index()
total_forecast.rename(columns={'forecasted_qty':'total_forecast_4w'}, inplace=True)

print(f"Forecast generated: {len(forecast_df)} rows")
print(forecast_df.head(8).to_string(index=False))

## 3. Lead Time Prediction per Product

In [ ]:
LT_FEATURES = ['po_month','po_dow','po_qty_log','product_cat_enc','warehouse_enc']

# สร้าง avg predicted lead time ต่อ product จาก PO history
po_lt = po.copy()
po_lt['po_month']    = po_lt['po_date'].dt.month
po_lt['po_dow']      = po_lt['po_date'].dt.dayofweek
po_lt['po_qty_log']  = np.log1p(po_lt['qty'])

# encode
cat_map = {c: i for i, c in enumerate(po_lt.merge(products,on='product_id')['product_taxonomies'].dropna().unique())}
wh_map  = {w: i for i, w in enumerate(po_lt['warehouse_id'].unique())}
po_lt = po_lt.merge(products[['product_id','product_taxonomies']], on='product_id', how='left')
po_lt['product_cat_enc'] = po_lt['product_taxonomies'].map(cat_map)
po_lt['warehouse_enc']   = po_lt['warehouse_id'].map(wh_map)

valid_po = po_lt.dropna(subset=LT_FEATURES)
if len(valid_po) > 0:
    valid_po = valid_po.copy()
    valid_po['predicted_lead_time'] = model_m2.predict(valid_po[LT_FEATURES])
    avg_lt = valid_po.groupby('product_id')['predicted_lead_time'].mean().reset_index()
    avg_lt.rename(columns={'predicted_lead_time':'avg_lead_time_days'}, inplace=True)
else:
    avg_lt = po_lt[['product_id']].drop_duplicates()
    avg_lt['avg_lead_time_days'] = 7.0  # fallback

print("Avg predicted lead time per product:")
print(avg_lt.head(8).to_string(index=False))

## 4. Safety Stock Calculation

In [ ]:
# Safety Stock = Z × σ_demand × √(lead_time)
demand_std = (
    feature_df.groupby(['store_id','product_id'])['qty_sold']
    .std().reset_index().rename(columns={'qty_sold':'demand_std_weekly'})
)
demand_std['demand_std_weekly'] = demand_std['demand_std_weekly'].fillna(1)

safety = demand_std.merge(avg_lt, on='product_id', how='left')
safety['avg_lead_time_days'] = safety['avg_lead_time_days'].fillna(7)
safety['lead_time_weeks']    = safety['avg_lead_time_days'] / 7
safety['safety_stock']       = (SERVICE_Z * safety['demand_std_weekly'] * np.sqrt(safety['lead_time_weeks'])).round(0)

print(f"Safety stock calculated for {len(safety)} product-store pairs")
print(safety.head(5).to_string(index=False))

## 5. Expiry Risk Engine

In [ ]:
# ── M3: Expiry Risk Score ────────────────────────────────────
REFERENCE = TODAY

po_risk = po.copy()
po_risk['days_to_expire'] = (po_risk['expire_date'] - REFERENCE).dt.days
po_risk = po_risk.merge(
    feature_df.groupby('product_id')['qty_sold'].mean().reset_index().rename(columns={'qty_sold':'avg_daily_sales'}),
    on='product_id', how='left'
)
po_risk['avg_daily_sales'] = po_risk['avg_daily_sales'].fillna(1) / 7

def expiry_risk(row):
    if row['days_to_expire'] <= 0:
        return 'EXPIRED', 0
    days_to_sell = row['qty'] / max(row['avg_daily_sales'], 0.1)
    ratio = days_to_sell / max(row['days_to_expire'], 1)
    if ratio > 1.2:   return 'HIGH', 0.20
    elif ratio > 1.0: return 'MEDIUM', 0.10
    else:             return 'LOW', 0.0

po_risk[['expiry_risk','recommended_discount']] = po_risk.apply(
    lambda r: pd.Series(expiry_risk(r)), axis=1)

high_risk = po_risk[po_risk['expiry_risk'].isin(['HIGH','EXPIRED'])]
print(f"High/Expired risk POs: {len(high_risk)}")
print(high_risk[['po_id','product_id','expire_date','days_to_expire','expiry_risk','recommended_discount']].head(8).to_string(index=False))

## 6. PO Recommendation Engine

In [ ]:
# ── Co-optimization: รวม forecast + stock + lead time → PO recs ──
current_stock_map = dict(zip(current_stock['store_id'], current_stock['current_stock']))

recs = []
for store in stores['store_id']:
    store_forecast = total_forecast[total_forecast['store_id'] == store]
    stock_now = current_stock_map.get(store, 0)

    for _, row in store_forecast.iterrows():
        prod_id       = row['product_id']
        forecast_4w   = row['total_forecast_4w']

        # lead time
        lt_row = avg_lt[avg_lt['product_id'] == prod_id]
        lead_days = lt_row['avg_lead_time_days'].values[0] if len(lt_row) else 7

        # safety stock
        ss_row = safety[(safety['store_id']==store) & (safety['product_id']==prod_id)]
        sstock = ss_row['safety_stock'].values[0] if len(ss_row) else 5

        # stock after 4 weeks (rough)
        stock_per_prod = stock_now / max(len(store_forecast), 1)
        projected_stock_4w = stock_per_prod - forecast_4w

        # gap
        required = forecast_4w + sstock
        gap = required - max(stock_per_prod, 0)

        if gap <= 0:
            urgency = "✅ No Order Needed"
            suggested_qty = 0
        else:
            suggested_qty = int(np.ceil(gap))
            order_by = TODAY + pd.Timedelta(days=max(0, 28 - lead_days - 3))
            days_until_order = (order_by - TODAY).days

            if days_until_order <= 2:    urgency = "🔴 Order Today"
            elif days_until_order <= 7:  urgency = "🟠 Order This Week"
            else:                        urgency = "🟡 Plan Ahead"

        # expiry constraint
        pr = po_risk[po_risk['product_id'] == prod_id]
        if len(pr) > 0 and pr['expiry_risk'].iloc[0] == 'HIGH':
            avg_daily = pr['avg_daily_sales'].iloc[0]
            dte = pr['days_to_expire'].iloc[0]
            max_safe_qty = int(avg_daily * dte * 0.8)
            suggested_qty = min(suggested_qty, max_safe_qty)
            urgency += " ⚠️ Expiry Limit"

        if suggested_qty > 0:
            recs.append({
                'store_id':       store,
                'product_id':     prod_id,
                'current_stock':  round(stock_per_prod, 0),
                'forecast_4w':    round(forecast_4w, 0),
                'safety_stock':   round(sstock, 0),
                'suggested_qty':  suggested_qty,
                'lead_time_days': round(lead_days, 1),
                'urgency':        urgency,
            })

po_recs = pd.DataFrame(recs).sort_values('urgency')
print(f"PO Recommendations generated: {len(po_recs)} items")
po_recs.head(15)

## 7. Final Output

In [ ]:
import os
os.makedirs("data/processed", exist_ok=True)

po_recs.to_csv("data/processed/po_recommendations.csv", index=False)

print("=" * 60)
print("  WEEKLY PO RECOMMENDATION SUMMARY")
print("=" * 60)
print(f"  Total items to order : {len(po_recs)}")
print(f"  Total suggested qty  : {po_recs['suggested_qty'].sum():,.0f} units")
print()
print("  By urgency:")
for urgency, grp in po_recs.groupby('urgency', sort=False):
    print(f"    {urgency:<35} {len(grp):>3} items")
print()
print("  ✅ Saved → data/processed/po_recommendations.csv")
print("  ✅ Ready to load into Streamlit dashboard")